# Benchmarking


In [ ]:
import os
from pathlib import Path
import re
import json
from collections import Counter
from datetime import datetime


# root directory of the project
PROJECT_ROOT = Path.cwd().parent
output_dir = PROJECT_ROOT / "data" / "output"
output_dir.mkdir(parents=True, exist_ok=True)
# input file path for the Shakespeare text
shakespeare_text_path = PROJECT_ROOT / "data/input" / "complete_shakespeare.txt"
github_shakespeare_text_path = PROJECT_ROOT / "data/input" / "github_shakespeare.txt"
# output file path for the word counts JSON
output_path = output_dir / "shakespeare_word_counts.json"


In [ ]:
def count_words_in_file(file_path: Path):
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"Input file not found: {path}")

    word_counts: dict[str, int] = {}

    with path.open("r", encoding="utf-8") as file:
        for line in file:
            for word in re.findall(r"[a-z']+", line.lower()):
                if word in word_counts:
                    word_counts[word] += 1
                else:
                    word_counts[word] = 1

    return dict(sorted(word_counts.items(),
                       key=lambda kv: kv[1],
                       reverse=True))

In [ ]:

def save_word_counts(word_counts: dict[str, int], output_dir: Path) -> Path:
    """
    Save word counts to JSON using a filename that includes
    the function name and the date/time it was run.
    """

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{save_word_counts.__name__}_{timestamp}.json"

    output_path = output_dir / filename

    with output_path.open("w", encoding="utf-8") as file:
        json.dump(word_counts, file, indent=4)

    return output_path

In [ ]:
def count_save_word_count(word_file_path: Path, output_dir: Path) -> Path:
    word_counts = count_words_in_file(word_file_path)
    return save_word_counts(word_counts, output_dir)

In [ ]:
%timeit -n 10 -r 10 count_save_word_count(word_file_path=shakespeare_text_path, output_dir=output_dir)


In [ ]:
# NOTE: use counter for greater performance


WORD_RE = re.compile(r"[a-z']+")

def count_words_counter(file_path: Path):
    counts = Counter()

    with Path(file_path).open("r", encoding="utf-8") as file:
        for line in file:
            counts.update(WORD_RE.findall(line.lower()))

    return counts

In [ ]:
word_counts = count_words_counter(shakespeare_text_path)
save_word_counts(word_counts, output_dir)

In [ ]:
%timeit -n 10 -r 10 count_save_word_counter(word_file_path=shakespeare_text_path, output_dir=output_dir)

In [ ]:

# NOTE: 
WORD_RE = re.compile(r"[a-z']+")

def counter_words_in_file_most_common(file_path: Path) -> dict[str, int]:
    counts = Counter()

    with Path(file_path).open("r", encoding="utf-8") as file:
        for line in file:
            counts.update(WORD_RE.findall(line.lower()))

    return dict(counts.most_common())

In [ ]:
word_counts = counter_words_in_file_most_common(shakespeare_text_path)
save_word_counts(word_counts, output_dir)

In [ ]:
%timeit -n 10 -r 10 counter_words_in_file_most_common(shakespeare_text_path)

In [ ]:
%prun count_words_in_file(shakespeare_text_path)